# Agente Mundial 2026 — LangGraph + Subgrafo + Human in the Loop

Arquitectura:
- **Grafo principal**: scheduler → agent → tools → subgrafo_partido (x partido) → human_review → send
- **Subgrafo**: por cada partido busca forma local+visitante en paralelo y genera el bloque de análisis

## 1. Instalación

In [1]:
%pip install -q langchain langchain-openai langgraph tavily-python requests


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Claves

In [2]:
import os

os.environ["AZURE_OPENAI_ENDPOINT"]       = "https://n8nprueba-resource.services.ai.azure.com"
os.environ["AZURE_OPENAI_API_KEY"]        = "Fhyf30hJicRBBXThBvTBLNfdNtxco39E3ld4ByG9h8VYM1RJCoMBJQQJ99CFACfhMk5XJ3w3AAAAACOGJMTZ"
os.environ["AZURE_OPENAI_DEPLOYMENT"]     = "gpt-4o-mini"      
os.environ["AZURE_OPENAI_API_VERSION"]    = "2024-02-15-preview"

os.environ["TAVILY_API_KEY"]              = "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"

os.environ["EMAIL_FROM"]                  = "alejandrobenitez91203@gmail.com"
os.environ["EMAIL_PASSWORD"]              = "tobp caof coxc xxpe"
os.environ["EMAIL_TO"]                    = "alejandrobenitez91203@gmail.com"

print("Claves configuradas.")

Claves configuradas.


## 3. Tools

In [3]:
import os
import smtplib
from datetime import date, timedelta
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
from tavily import TavilyClient
from langchain.tools import tool


def _tavily_search(query: str, max_results: int = 5) -> str:
    client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
    resp = client.search(query=query, max_results=max_results, search_depth="advanced")
    results = resp.get("results", [])
    if not results:
        return "Sin resultados."
    return "\n".join(
        f"[{r['title']}]\n{r['content']}\nFuente: {r['url']}\n"
        for r in results
    )


@tool
def get_matches_today() -> str:
    """Busca los partidos del Mundial 2026 programados para hoy."""
    today = date.today().strftime("%B %d %Y")
    return _tavily_search(f"FIFA World Cup 2026 matches today {today} schedule kickoff time")


@tool
def get_next_matches() -> str:
    """Busca todos los partidos del Mundial 2026 de los próximos 3 días, día a día."""
    results = []
    for i in range(1, 4):
        day = date.today() + timedelta(days=i)
        text = _tavily_search(
            f"FIFA World Cup 2026 all matches {day.strftime('%B %d %Y')} complete schedule kickoff times",
            max_results=8
        )
        results.append(f"=== {day.isoformat()} ===\n{text}")
    return "\n\n".join(results)


@tool
def get_team_form(team_name: str) -> str:
    """Busca el estado de forma reciente de una selección. team_name en español o inglés."""
    return _tavily_search(
        f"{team_name} seleccion nacional forma reciente resultados Mundial 2026 jugadores clave",
        max_results=3
    )


@tool
def write_matches_txt(content: str, subject: str = "") -> str:
    """Escribe el análisis completo en partidos.txt. Si ya existe lo sobreescribe."""
    with open("partidos.txt", "w", encoding="utf-8") as f:
        f.write(content)
    subject_final = subject or f"Mundial 2026 — {date.today().strftime('%d/%m/%Y')}"
    with open("email_subject.txt", "w", encoding="utf-8") as f:
        f.write(subject_final)
    return f"partidos.txt escrito ({len(content)} chars). Asunto: '{subject_final}'."


@tool
def send_email_with_file(filepath: str = "partidos.txt") -> str:
    """Envía partidos.txt por email como cuerpo y adjunto."""
    from_addr = os.environ["EMAIL_FROM"]
    to_addr   = os.environ["EMAIL_TO"]
    password  = os.environ["EMAIL_PASSWORD"].replace(" ", "")
    try:
        subject = open("email_subject.txt", encoding="utf-8").read().strip()
    except FileNotFoundError:
        subject = f"Mundial 2026 — {date.today().strftime('%d/%m/%Y')}"
    try:
        body_text = open(filepath, encoding="utf-8").read()
    except FileNotFoundError:
        return f"ERROR: '{filepath}' no existe."
    if not body_text.strip():
        return "ERROR: partidos.txt vacío."
    msg = MIMEMultipart()
    msg["From"]    = from_addr
    msg["To"]      = to_addr
    msg["Subject"] = subject
    msg.attach(MIMEText(body_text, "plain", "utf-8"))
    with open(filepath, "rb") as f:
        part = MIMEBase("application", "octet-stream")
        part.set_payload(f.read())
    encoders.encode_base64(part)
    part.add_header("Content-Disposition", f'attachment; filename="{filepath}"')
    msg.attach(part)
    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=10) as server:
            server.login(from_addr, password)
            server.sendmail(from_addr, to_addr, msg.as_string())
        return f"Email enviado a {to_addr}. Asunto: '{subject}'."
    except smtplib.SMTPAuthenticationError:
        return "ERROR auth: verifica verificación en 2 pasos y contraseña de aplicación."
    except Exception as e:
        return f"ERROR: {e}"


TOOLS = [get_matches_today, get_next_matches, get_team_form, write_matches_txt, send_email_with_file]
TOOLS_BY_NAME = {t.name: t for t in TOOLS}
print("Tools:", list(TOOLS_BY_NAME.keys()))

Tools: ['get_matches_today', 'get_next_matches', 'get_team_form', 'write_matches_txt', 'send_email_with_file']


## 4. Subgrafo — análisis por partido

Por cada partido: busca forma de local y visitante en paralelo (ThreadPoolExecutor), luego genera el bloque de análisis con el LLM.

In [4]:
from typing import Annotated, Optional
from concurrent.futures import ThreadPoolExecutor
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage, BaseMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from typing_extensions import TypedDict


# ── LLM compartido ────────────────────────────────────────────────
llm = AzureChatOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    temperature=0.7
)


# ────────────────────────────────────────────────────────────────
# SUBGRAFO: analiza un partido individual
# Estado propio — no comparte estado con el grafo principal
# ────────────────────────────────────────────────────────────────
class MatchState(TypedDict):
    home:        str            # equipo local
    away:        str            # equipo visitante
    fixture_info: str           # fecha, hora, estadio
    home_form:   str            # resultado de get_team_form(home)
    away_form:   str            # resultado de get_team_form(away)
    analysis:    str            # bloque final generado por el LLM


def subnode_fetch_form(state: MatchState) -> MatchState:
    """
    Busca el estado de forma de local y visitante en paralelo.
    Las dos llamadas a Tavily se lanzan simultáneamente con ThreadPoolExecutor.
    """
    print(f"  [subgrafo: fetch_form] {state['home']} vs {state['away']}")
    with ThreadPoolExecutor(max_workers=2) as executor:
        fut_home = executor.submit(get_team_form.invoke, {"team_name": state["home"]})
        fut_away = executor.submit(get_team_form.invoke, {"team_name": state["away"]})
        home_form = fut_home.result()
        away_form = fut_away.result()
    return {"home_form": home_form, "away_form": away_form}


def subnode_generate_analysis(state: MatchState) -> MatchState:
    """
    Con la forma de ambos equipos ya disponible, llama al LLM
    para generar el bloque de análisis de apuestas del partido.
    """
    print(f"  [subgrafo: generate_analysis] {state['home']} vs {state['away']}")
    prompt = f"""Genera el bloque de análisis de apuestas para este partido del Mundial 2026.
Usa el estilo de Paolo Maldini: elegante, directo, datos reales.
Texto plano, sin markdown.

PARTIDO: {state['home']} vs {state['away']}
INFO: {state['fixture_info']}

FORMA {state['home'].upper()}:
{state['home_form']}

FORMA {state['away'].upper()}:
{state['away_form']}

FORMATO DE SALIDA:
{state['home']} vs {state['away']}
Fecha: [DD/MM/YYYY a las HH:MM hora España CEST (UTC+2)] | [estadio, ciudad]

[2 líneas de contexto: grupo, qué se juegan]

Claves para apostar:
- 1X2: [quién gana — confianza alta/media/baja]
- Más/Menos 2.5 goles: [razonado]
- Ambos marcan: [sí/no, razonado]
- Handicap: [si hay favorito claro]
- APUESTA RECOMENDADA: [mejor valor esperado]
"""
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"analysis": response.content}


# ── Construcción del subgrafo ─────────────────────────────────────
match_graph_builder = StateGraph(MatchState)
match_graph_builder.add_node("fetch_form", subnode_fetch_form)
match_graph_builder.add_node("generate_analysis", subnode_generate_analysis)
match_graph_builder.set_entry_point("fetch_form")
match_graph_builder.add_edge("fetch_form", "generate_analysis")
match_graph_builder.add_edge("generate_analysis", END)
match_subgraph = match_graph_builder.compile()

print("Subgrafo compilado. Nodos:", list(match_graph_builder.nodes.keys()))

Subgrafo compilado. Nodos: ['fetch_form', 'generate_analysis']


## 5. Grafo principal

In [5]:
import re


# ── Estado del grafo principal ────────────────────────────────────
class AgentState(TypedDict):
    messages:  Annotated[list[BaseMessage], add_messages]
    fixtures:  list[dict]   # partidos extraídos por el agente [{home, away, fixture_info}]
    analyses:  list[str]    # bloques generados por el subgrafo, uno por partido
    approved:  bool         # decisión del humano


llm_with_tools = llm.bind_tools([get_matches_today, get_next_matches])

SYSTEM_PROMPT = """Eres un agente que obtiene los partidos del Mundial 2026.

PASO 1: Llama a get_matches_today.
PASO 2a: Si hay partidos hoy, extrae la lista de partidos y PARA.
PASO 2b: Si NO hay partidos (respuesta sin fixtures), llama a get_next_matches, extrae la lista y PARA.

Cuando tengas la lista de partidos, responde SOLO con un JSON con esta estructura y nada más:
{
  "fixtures": [
    {"home": "Nombre local", "away": "Nombre visitante", "fixture_info": "fecha hora estadio"},
    ...
  ]
}"""


# ── Nodo agent ────────────────────────────────────────────────────
def node_agent(state: AgentState) -> AgentState:
    print("[nodo: agent]")
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


# ── Nodo tools — ejecución paralela ──────────────────────────────
def node_tools(state: AgentState) -> AgentState:
    from concurrent.futures import ThreadPoolExecutor, as_completed
    last_message = state["messages"][-1]
    tool_calls = last_message.tool_calls

    def run_tool(tc):
        print(f"[nodo: tools] → {tc['name']}({list(tc['args'].keys())})")
        result = TOOLS_BY_NAME[tc["name"]].invoke(tc["args"])
        return ToolMessage(content=str(result), tool_call_id=tc["id"])

    if len(tool_calls) == 1:
        return {"messages": [run_tool(tool_calls[0])]}

    tool_messages = [None] * len(tool_calls)
    with ThreadPoolExecutor(max_workers=min(len(tool_calls), 8)) as executor:
        futures = {executor.submit(run_tool, tc): i for i, tc in enumerate(tool_calls)}
        for future in as_completed(futures):
            tool_messages[futures[future]] = future.result()
    return {"messages": tool_messages}


# ── Nodo parse_fixtures: extrae la lista de partidos del JSON ─────
def node_parse_fixtures(state: AgentState) -> AgentState:
    import json
    print("[nodo: parse_fixtures]")
    last = state["messages"][-1]
    try:
        # Extraer JSON de la respuesta del LLM
        text = last.content if hasattr(last, "content") else str(last)
        match = re.search(r"\{.*\}", text, re.DOTALL)
        data = json.loads(match.group()) if match else {}
        fixtures = data.get("fixtures", [])
    except Exception as e:
        print(f"  [parse_fixtures] Error parseando JSON: {e}")
        fixtures = []
    print(f"  [parse_fixtures] {len(fixtures)} partidos encontrados")
    return {"fixtures": fixtures}


# ── Nodo analyze_matches: invoca el subgrafo por cada partido ─────
# Los subgrafos se ejecutan secuencialmente (uno por partido).
# Dentro de cada subgrafo las dos búsquedas de forma van en paralelo.
def node_analyze_matches(state: AgentState) -> AgentState:
    print("[nodo: analyze_matches]")
    analyses = []
    for fixture in state["fixtures"]:
        result = match_subgraph.invoke({
            "home":         fixture["home"],
            "away":         fixture["away"],
            "fixture_info": fixture["fixture_info"],
            "home_form":    "",
            "away_form":    "",
            "analysis":     ""
        })
        analyses.append(result["analysis"])
    return {"analyses": analyses}


# ── Nodo write_report: une todos los bloques y escribe el TXT ─────
def node_write_report(state: AgentState) -> AgentState:
    print("[nodo: write_report]")
    today = date.today().strftime("%d/%m/%Y")
    separator = "\n" + "-" * 48 + "\n"
    body = separator.join(state["analyses"])
    full_text = f"MUNDIAL 2026 — {today}\n" + "=" * 48 + "\n\n" + body

    fixtures = state["fixtures"]
    if len(fixtures) == 1:
        subject = f"Mundial 2026 - {fixtures[0]['home']} vs {fixtures[0]['away']} - {today}"
    else:
        subject = f"Mundial 2026 - {len(fixtures)} partidos - {today}"

    write_matches_txt.invoke({"content": full_text, "subject": subject})
    print(f"  [write_report] TXT escrito. Asunto: '{subject}'")
    return {}


# ── Nodo human_review ─────────────────────────────────────────────
def node_human_review(state: AgentState) -> AgentState:
    print("\n[nodo: human_review] — Grafo pausado.")
    try:
        print("\n" + "=" * 50)
        print(open("partidos.txt", encoding="utf-8").read())
        print("=" * 50)
    except FileNotFoundError:
        print("  partidos.txt no encontrado.")
    return state


# ── Nodo send ─────────────────────────────────────────────────────
def node_send(state: AgentState) -> AgentState:
    if state.get("approved", False):
        print("[nodo: send] Aprobado — enviando email...")
        print(f"[nodo: send] {send_email_with_file.invoke({'filepath': 'partidos.txt'})}")
    else:
        print("[nodo: send] Rechazado — email cancelado.")
    return state


# ── Condición: agent sigue o pasa a parse ────────────────────────
def should_continue(state: AgentState) -> str:
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"
    return "parse_fixtures"


# ── Condición: hay fixtures para analizar o no ───────────────────
def has_fixtures(state: AgentState) -> str:
    return "analyze" if state.get("fixtures") else "no_fixtures"


# ── Nodo fallback: sin partidos ───────────────────────────────────
def node_no_fixtures(state: AgentState) -> AgentState:
    print("[nodo: no_fixtures] No se encontraron partidos.")
    return state


# ── Construcción del grafo principal ─────────────────────────────
memory = MemorySaver()
gb = StateGraph(AgentState)

gb.add_node("agent",           node_agent)
gb.add_node("tools",           node_tools)
gb.add_node("parse_fixtures",  node_parse_fixtures)
gb.add_node("analyze_matches", node_analyze_matches)   # invoca el subgrafo
gb.add_node("write_report",    node_write_report)
gb.add_node("human_review",    node_human_review)
gb.add_node("send",            node_send)
gb.add_node("no_fixtures",     node_no_fixtures)

gb.set_entry_point("agent")
gb.add_conditional_edges("agent", should_continue, {"tools": "tools", "parse_fixtures": "parse_fixtures"})
gb.add_edge("tools", "agent")
gb.add_conditional_edges("parse_fixtures", has_fixtures, {"analyze": "analyze_matches", "no_fixtures": "no_fixtures"})
gb.add_edge("analyze_matches", "write_report")
gb.add_edge("write_report",    "human_review")
gb.add_edge("human_review",    "send")
gb.add_edge("send",            END)
gb.add_edge("no_fixtures",     END)

graph = gb.compile(checkpointer=memory, interrupt_before=["human_review"])
print("Grafo principal compilado. Nodos:", list(gb.nodes.keys()))

Grafo principal compilado. Nodos: ['agent', 'tools', 'parse_fixtures', 'analyze_matches', 'write_report', 'human_review', 'send', 'no_fixtures']


## 6. Ejecutar — Fase 1: análisis

In [6]:
config = {"configurable": {"thread_id": "mundial_2026"}}

print("Fase 1: arrancando agente...\n")
graph.invoke(
    {
        "messages":  [HumanMessage(content="Procesa los partidos del Mundial 2026 de hoy.")],
        "fixtures":  [],
        "analyses":  [],
        "approved":  False
    },
    config=config
)
print("\nGrafo pausado. Revisa el análisis y ejecuta la celda 7.")

Fase 1: arrancando agente...

[nodo: agent]
[nodo: tools] → get_matches_today([])
[nodo: agent]
[nodo: tools] → get_next_matches([])
[nodo: agent]
[nodo: parse_fixtures]
  [parse_fixtures] 2 partidos encontrados
[nodo: analyze_matches]
  [subgrafo: fetch_form] Mexico vs South Africa
  [subgrafo: generate_analysis] Mexico vs South Africa
  [subgrafo: fetch_form] South Korea vs Czechia
  [subgrafo: generate_analysis] South Korea vs Czechia
[nodo: write_report]
  [write_report] TXT escrito. Asunto: 'Mundial 2026 - 2 partidos - 09/06/2026'

Grafo pausado. Revisa el análisis y ejecuta la celda 7.


## 7. Human in the Loop — aprueba o rechaza

Cambia `approved = True` para enviar el email, `False` para cancelar.

In [7]:
approved = True

print(f"Decisión: {'ENVIAR' if approved else 'CANCELAR'}\n")
graph.invoke({"approved": approved}, config=config)
print("\nFlujo completado.")

Decisión: ENVIAR

[nodo: agent]
[nodo: parse_fixtures]
  [parse_fixtures] 2 partidos encontrados
[nodo: analyze_matches]
  [subgrafo: fetch_form] Mexico vs South Africa
  [subgrafo: generate_analysis] Mexico vs South Africa
  [subgrafo: fetch_form] South Korea vs Czechia
  [subgrafo: generate_analysis] South Korea vs Czechia
[nodo: write_report]
  [write_report] TXT escrito. Asunto: 'Mundial 2026 - 2 partidos - 09/06/2026'

Flujo completado.


## 8. Debug

In [8]:
state = graph.get_state(config)
print(f"Partidos encontrados: {len(state.values.get('fixtures', []))}")
print(f"Análisis generados:   {len(state.values.get('analyses', []))}")
print(f"Aprobado:             {state.values.get('approved')}")
print()
for i, msg in enumerate(state.values["messages"]):
    tipo = type(msg).__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"[{i}] {tipo} → tool_calls: {[tc['name'] for tc in msg.tool_calls]}")
    else:
        preview = str(msg.content)[:80].replace("\n", " ")
        print(f"[{i}] {tipo} → {preview}...")

Partidos encontrados: 2
Análisis generados:   2
Aprobado:             True

[0] HumanMessage → Procesa los partidos del Mundial 2026 de hoy....
[1] AIMessage → tool_calls: ['get_matches_today']
[2] ToolMessage → [World Cup 2026 fixture schedule and UK kick-off times: Day-by-day breakdown of ...
[3] AIMessage → tool_calls: ['get_next_matches']
[4] ToolMessage → === 2026-06-10 === [2026 FIFA World Cup Schedule: Kickoff times, dates, fixture ...
[5] AIMessage → ```json {   "fixtures": [     {       "home": "Mexico",       "away": "South Afr...
[6] AIMessage → ```json {   "fixtures": [     {       "home": "Mexico",       "away": "South Afr...
